# Executive Summary

This Healthcare Patient Analysis project examines patient's record to uncover patterns in **patient demographics**, **disease severity**, **treatment costs**, **healthcare utilization**, **doctor performance**, and **hospital efficiency**. The analysis aims to help healthcare organizations improve resource allocation, manage treatment costs, and enhance patient care outcomes through data-driven insights.
    
    Key Healthcare Findings
- Total medicine expenditure across all patients reached **$9.22 million**.
- **Asthma** emerged as one of the most frequently diagnosed conditions across the dataset.
- Average patient visits ranged from **5 to 7 visits** annually for chronic and cardiac conditions, indicating ongoing treatment requirements.
- High-repeat-visit patients were predominantly associated with *Diabetes, Hypertension, Arthritis, and Coronary Artery Disease, particularly within older age groups*.
- Patients with **severe conditions** consistently incurred higher medicine costs than **mild and moderate** cases.

In [49]:
import warnings
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import plotly.io as pio
pio.renderers.default = 'notebook_connected'

In [50]:
warnings.filterwarnings('ignore')
df = pd.read_excel('Patient_Healthcare_Analysis.xlsx')

## Understanding & Data Preperation

    Modify Column Names

In [51]:
#   Displaying Columns Name
df.columns

Index(['PatientID', 'PatientName', 'Name', 'Gender', 'Age', 'BloodGroup',
       'Insurance', 'MedicineID', 'MedicineName', 'Quantity', 'cost_price',
       'Disease', 'Disease Category', 'Severity Level', 'Diagnosis Date',
       'Doctor Name', 'Doctor Specialty', 'Hospital/Clinic',
       'Number of Visits (FY 2024)', 'Last Visit Date', 'Notes'],
      dtype='str')

- Now i am cleaning the column names to make them easier to work with. 
- I will convert them to lowercase, replace spaces with underscores, and remove any special characters if necessary. 
- This will help me avoid issues when referencing the columns in my code.

In [52]:
df.columns = df.columns.str.strip().str.lower().str.replace(' ', '_')
df.rename(columns={'number_of_visits_(fy_2024)': 'number_of_visits'}, inplace=True)

In [53]:
#   Display After Modify Column Names
df.columns

Index(['patientid', 'patientname', 'name', 'gender', 'age', 'bloodgroup',
       'insurance', 'medicineid', 'medicinename', 'quantity', 'cost_price',
       'disease', 'disease_category', 'severity_level', 'diagnosis_date',
       'doctor_name', 'doctor_specialty', 'hospital/clinic',
       'number_of_visits', 'last_visit_date', 'notes'],
      dtype='str')

    Understanding Data

In [54]:
#   First 5 Rows
df.head()

,patientid,patientname,name,gender,age,bloodgroup,insurance,medicineid,medicinename,quantity,...,disease,disease_category,severity_level,diagnosis_date,doctor_name,doctor_specialty,hospital/clinic,number_of_visits,last_visit_date,notes
0,P048,Patient_48,Paula Campos,Male,32,A-,LifeShield,M058,Medicine_58,3,...,Arthritis,Chronic,Severe,2023-06-17,Dr. P. Jain,Rheumatology,City Care Hospital,8,2023-12-07,Stable on current treatment
1,P001,Patient_1,Laura Evans,Male,19,AB+,HealthSecure,M061,Medicine_61,5,...,Migraine,Neurology,Moderate,2023-07-01,Dr. G. Chatterjee,Neurology,Fortune Care,4,2023-10-02,Follow-up in 3 months
2,P028,Patient_28,Daniel Bauer,Female,75,O+,LifeShield,M091,Medicine_91,5,...,Allergic Rhinitis,Respiratory,Mild,2024-01-18,Dr. R. Nair,ENT,Fortune Care,6,2024-02-18,Physiotherapy recommended
3,P011,Patient_11,Zachary Dudley,Female,56,AB+,MediCare Plus,M084,Medicine_84,3,...,Asthma,Respiratory,Mild,2024-01-27,Dr. N. Verma,Pulmonology,Greenfield Medical Centre,5,2024-02-07,Medication adjusted
4,P016,Patient_16,Eric Keith,Male,72,B-,MediCare Plus,M025,Medicine_25,4,...,Allergic Rhinitis,Respiratory,Mild,2023-04-05,Dr. A. Bansal,ENT,Fortune Care,5,2023-12-28,Stable on current treatment


    Check Empty Values

In [55]:
print(f'Null values in the dataset are: {df.isnull().sum().sum()}')

Null values in the dataset are: 0


    Check Duplicate Values

In [56]:
print(f'Duplicate entries in the dataset are: {df.duplicated().sum()}')

Duplicate entries in the dataset are: 0


    Gathering Information

In [57]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 1000 entries, 0 to 999
Data columns (total 21 columns):
 #   Column            Non-Null Count  Dtype         
---  ------            --------------  -----         
 0   patientid         1000 non-null   str           
 1   patientname       1000 non-null   str           
 2   name              1000 non-null   str           
 3   gender            1000 non-null   str           
 4   age               1000 non-null   int64         
 5   bloodgroup        1000 non-null   str           
 6   insurance         1000 non-null   str           
 7   medicineid        1000 non-null   str           
 8   medicinename      1000 non-null   str           
 9   quantity          1000 non-null   int64         
 10  cost_price        1000 non-null   int64         
 11  disease           1000 non-null   str           
 12  disease_category  1000 non-null   str           
 13  severity_level    1000 non-null   str           
 14  diagnosis_date    1000 non-null   da

In [58]:
df.describe()

,age,quantity,cost_price,diagnosis_date,number_of_visits,last_visit_date
count,1000.000000,1000.000000,1000.000000,1000,1000.000000,1000
mean,47.360000,3.108000,2964.700000,2023-09-27 05:55:40.800000,5.180000,2023-12-30 05:24:00
min,18.000000,1.000000,193.000000,2023-04-01 00:00:00,1.000000,2023-04-20 00:00:00
25%,32.000000,2.000000,2021.500000,2023-06-25 00:00:00,3.000000,2023-11-08 00:00:00
50%,50.500000,3.000000,3077.500000,2023-09-25 12:00:00,4.000000,2024-01-24 00:00:00
75%,61.000000,4.000000,4354.500000,2023-12-27 00:00:00,8.000000,2024-03-08 00:00:00
max,79.000000,5.000000,4979.000000,2024-03-31 00:00:00,12.000000,2024-03-31 00:00:00
std,17.464691,1.405102,1414.340549,NaN,3.143482,NaN


In [59]:
df.describe(include='object')

,patientid,patientname,name,gender,bloodgroup,insurance,medicineid,medicinename,disease,disease_category,severity_level,doctor_name,doctor_specialty,hospital/clinic,notes
count,1000,1000,1000,1000,1000,1000,1000,1000,1000,1000,1000,1000,1000,1000,1000
unique,50,50,50,2,8,5,100,100,10,5,3,28,7,9,8
top,P048,Patient_48,Paula Campos,Male,A-,PrimeHealth,M058,Medicine_58,Asthma,Chronic,Mild,Dr. M. Agarwal,Cardiology,Lotus Hospital,Diagnostic tests advised
freq,20,20,20,660,200,300,10,10,111,396,539,57,206,120,140


    Create/Add Columns

In [60]:
# Calculate total medicine cost   
df['total_medicine_cost'] = df['quantity'] * df['cost_price']

In [61]:
# Create age groups
df['age_group'] = pd.cut(df['age'],bins=[18, 30, 45, 60, 80],labels=['18-30', '31-45', '46-60', '61-80'])

## Data Analysis

### 1) Revenue & Cost Signals

    Total medicine cost across all patients

In [62]:
#   Total medicine cost across all patients
print(f'Total medicine cost across all patients: ${df["total_medicine_cost"].sum():,}')

Total medicine cost across all patients: $9,216,626


    High-cost patient segments (top 10% by medicine cost)

In [63]:
threshold = df['total_medicine_cost'].quantile(0.90)

high_cost = df[df['total_medicine_cost'] >= threshold]

segment_analysis = high_cost.groupby(['disease_category','severity_level']).agg(
    patient_count=('patientid','count'),
    avg_cost=('total_medicine_cost','mean'),
    total_cost=('total_medicine_cost','sum')
).reset_index()

segment_analysis['avg_cost'] = segment_analysis['avg_cost'].round(1)
segment_analysis = segment_analysis.sort_values(by='total_cost',ascending=False)

segment_analysis

,disease_category,severity_level,patient_count,avg_cost,total_cost
5,Chronic,Mild,27,22688.5,612590
6,Chronic,Moderate,19,21880.3,415726
10,Respiratory,Mild,14,21964.8,307507
1,Acute,Moderate,8,21435.4,171483
0,Acute,Mild,8,20824.5,166596
11,Respiratory,Moderate,6,21524.0,129144
8,Neurology,Mild,5,20046.0,100230
9,Neurology,Moderate,4,21953.8,87815
7,Chronic,Severe,3,23426.7,70280
4,Cardiac,Moderate,3,20739.7,62219


In [64]:
fig = go.Figure()

colors = {
    'Mild':'#43A047',
    'Moderate':'#FB8C00',
    'Severe':'#E53935'}

for severity in segment_analysis['severity_level'].unique():

    data = segment_analysis[segment_analysis['severity_level'] == severity]

    fig.add_trace(go.Bar(
        x=data['disease_category'],
        y=data['total_cost'],

        name=f'<b>{severity}</b>',

        text=(data['total_cost']/1000).round(1),
        texttemplate='<b>$%{text}K</b>',
        textposition='outside',

        marker=dict(
            color=colors[severity],
            line=dict(color='white', width=1.5)),

        customdata=data[['patient_count','avg_cost']],
        hovertemplate=
        f'<b>Severity:</b> {severity}<br>' +
        '<b>Patients:</b> %{customdata[0]:,}<br>' +
        '<b>Average Cost:</b> $%{customdata[1]:,.0f}<br>' +
        '<b>Total Cost:</b> $%{y:,.0f}<br><extra></extra>'
    ))
fig.update_layout(
    title=dict(
        text='<b>High-Cost Patient Segments (Top 10% Medicine Cost)</b>',
        x=0.5,
        font=dict(size=24)),
        
    xaxis=dict(
        title=dict(
            text='<b>Disease Category</b>',
            font=dict(size=16)),
        tickfont=dict(size=14)),

    yaxis=dict(
        title=dict(
            text='<b>Total Medicine Cost ($)</b>',
            font=dict(size=16)),
        tickfont=dict(size=14)),

    template='simple_white',
    hovermode='x unified',
    barmode='group',
    bargap=0.22,
    margin=dict(l=90, r=0, t=100, b=80),

    legend=dict(
        title=dict(text='<b>Severity Level</b>', font=dict(size=14)),
        x=0.83,
        y=1.14,
        bgcolor='rgba(255,255,255,0.7)',
        bordercolor='lightgray',
        borderwidth=1
    ))
fig.show()

    Correlation heatmap (Age, Quantity, cost_price, Visits, Total_Medicine_Cost)

In [65]:
corr_data = df[['age', 'quantity', 'cost_price', 'number_of_visits', 'total_medicine_cost']].corr().round(2)

corr_data

,age,quantity,cost_price,number_of_visits,total_medicine_cost
age,1.00,-0.03,0.04,0.05,0.02
quantity,-0.03,1.00,0.00,0.03,0.64
cost_price,0.04,0.00,1.00,-0.08,0.70
number_of_visits,0.05,0.03,-0.08,1.00,-0.04
total_medicine_cost,0.02,0.64,0.70,-0.04,1.00


In [66]:
fig = go.Figure()

fig.add_trace(go.Heatmap(

    x=corr_data.index,
    y=corr_data.columns,
    z=corr_data.values,

    colorscale= 'RdBu',

    zmin=-1,
    zmax=1,

    text=corr_data.values,
    texttemplate='<b>%{text}</b>',

    hovertemplate=
    '<b>X Variable:</b> %{x}<br>' +
    '<b>Y Variable:</b> %{y}<br>' +
    '<b>Correlation:</b> %{z:.2f}<extra></extra>',

    colorbar=dict(
        title='<b>Correlation</b>')
    ))
fig.update_layout(

    title=dict(
        text='<b>Healthcare Correlation Heatmap</b>',
        x=0.5,
        font=dict(size=24)),

    xaxis=dict(
        tickfont=dict(size=14),
        side='bottom'),

    yaxis=dict(
        tickfont=dict(size=14),
        tickangle=-45),

    template='simple_white',
    margin=dict(l=120, r=0, t=100, b=80))

fig.show()

### 2) Patient Demographics & Distribution

    Age distribution by Gender + Age Group (histogram / violin)

In [67]:
age_gender = df.groupby(['age_group', 'gender']).size().reset_index(name='patients')

age_gender

,age_group,gender,patients
0,18-30,Female,20
1,18-30,Male,180
2,31-45,Female,140
3,31-45,Male,100
4,46-60,Female,100
5,46-60,Male,160
6,61-80,Female,80
7,61-80,Male,200


In [68]:
fig = go.Figure()

male_data = age_gender[age_gender['gender'] == 'Male']

fig.add_trace(go.Bar(
    x=male_data['age_group'],
    y=male_data['patients'],

    name='<b>Male</b>',

    text=male_data['patients'],
    texttemplate='<b>%{text}</b>',
    textposition='outside',

    marker=dict(
        color='#1E88E5',
        line=dict(color='white', width=1.5)),

    hovertemplate=
    '<b>Gender:</b> Male<br>' +
    '<b>Total Patients:</b> %{y}<br><extra></extra>'
    ))
female_data = age_gender[age_gender['gender'] == 'Female']

fig.add_trace(go.Bar(
    x=female_data['age_group'],
    y=female_data['patients'],

    name='<b>Female</b>',

    text=female_data['patients'],
    texttemplate='<b>%{text}</b>',
    textposition='outside',

    marker=dict(
        color='#E91E63',
        line=dict(color='white', width=1.5)),

    hovertemplate=
    '<b>Gender:</b> Female<br>' +
    '<b>Total Patients:</b> %{y}<extra></extra>'
    ))
fig.update_layout(
    title=dict(
        text='<b>Age Distribution by Gender & Age Group</b>',
        x=0.5,
        font=dict(size=23)),

    xaxis=dict(
        title=dict(
            text='<b>Age Groups</b>',
            font=dict(size=18)),
        tickfont=dict(size=14)),

    yaxis=dict(
        title=dict(
            text='<b>Number of Patients</b>',
            font=dict(size=18)),
        tickfont=dict(size=14),
        showgrid=True,
        gridcolor='rgba(0,0,0,0.15)'
        ),

    barmode='group',
    bargap=0.22,
    template='simple_white',
    hovermode='x unified',
    margin=dict(t=110, b=80, l=85, r=10),

    legend=dict(
        title=dict(text='<b>Gender</b>', font=dict(size=16)),
        orientation='v',
        x=0.88,
        y=1.22,
        ))
fig.show()

    Blood Group and Insurance coverage distribution

In [69]:
blood_dist = df['bloodgroup'].value_counts().reset_index()
blood_dist.columns = ['bloodgroup', 'patients']

blood_dist

,bloodgroup,patients
0,A-,200
1,AB+,200
2,B-,160
3,O-,140
4,AB-,140
5,O+,100
6,A+,40
7,B+,20


In [70]:
insurance_dist = df['insurance'].value_counts().reset_index()

insurance_dist.columns = ['insurance', 'patients']

insurance_dist

,insurance,patients
0,PrimeHealth,300
1,HealthSecure,220
2,MediCare Plus,220
3,LifeShield,160
4,CareFirst,100


In [71]:
fig = make_subplots(
    rows=1,
    cols=2,
    specs=[[{'type':'domain'}, {'type':'domain'}]],
    subplot_titles=(
        '<b>Blood Group Distribution</b>',
        '<b>Insurance Coverage Distribution</b>'
    ))
fig.add_trace(go.Pie(
    labels=blood_dist['bloodgroup'],
    values=blood_dist['patients'],

    hole=0.5,

    textinfo='percent+label',
    textposition='inside',
    textfont=dict(size=12, color='white'),

    marker=dict(
        colors=blood_dist['bloodgroup'],
        line=dict(color='white', width=2)),

    hovertemplate=
    '<b>Blood Group:</b> %{label}<br>' +
    '<b>Patients:</b> %{value}<br>' +
    '<b>Percentage:</b> %{percent}<extra></extra>'
), row=1, col=1)

fig.add_trace(go.Pie(
    labels=insurance_dist['insurance'],
    values=insurance_dist['patients'],

    hole=0.5,

    textinfo='percent+label',
    textposition='inside',
    textfont=dict(size=12, color='white'),

    marker=dict(
        colors=['#0A2540', '#1E88E5', '#8E24AA', '#6D4C41', '#E53935'],
        line=dict(
            color='white', width=2)),

    hovertemplate=
    '<b>Insurance:</b> %{label}<br>' +
    '<b>Patients:</b> %{value}<br>' +
    '<b>Percentage:</b> %{percent}<extra></extra>'
), row=1, col=2)

fig.update_layout(
    annotations=[
        dict(
            text='<b>Blood<br>Groups</b>',
            x=0.225,
            y=0.43,
            showarrow=False,
            font=dict(size=18)),
        dict(
            text='<b>Insurance<br>Coverage</b>',
            x=0.777,
            y=0.43,
            showarrow=False,
            font=dict(size=18))],

    title=dict(
        text='<b>Patient Blood Group & Insurance Distribution</b>',
        x=0.5,
        y=0.93,
        font=dict(size=24)),

    template='simple_white',
    margin=dict(t=150,b=50),

    legend=dict(
        orientation='h',
        y=1.25,
        x=0.2
    ))
fig.show()

    Patient volume by Disease Category and Severity Level (stacked bar / heatmap)

In [72]:
severity_analysis = df.groupby(['disease_category', 'severity_level']).size().reset_index(name='patients')

severity_analysis

,disease_category,severity_level,patients
0,Acute,Mild,84
1,Acute,Moderate,71
2,Acute,Severe,19
3,Cardiac,Mild,56
4,Cardiac,Moderate,36
5,Cardiac,Severe,13
6,Chronic,Mild,212
7,Chronic,Moderate,137
8,Chronic,Severe,47
9,Neurology,Mild,64


In [73]:
fig = go.Figure()

mild_data = severity_analysis[severity_analysis['severity_level'] == 'Mild']

fig.add_trace(go.Bar(
    x=mild_data['disease_category'],
    y=mild_data['patients'],

    name='<b>Mild</b>',
    marker=dict(color='cyan'),

    text=mild_data['patients'],
    texttemplate='<b>%{text}</b>',
    textposition='inside',

    hovertemplate=
    '<b>Severity:</b> Mild<br>' +
    '<b>Patients:</b> %{y}<extra></extra>'
    ))
moderate_data = severity_analysis[severity_analysis['severity_level'] == 'Moderate']

fig.add_trace(go.Bar(
    x=moderate_data['disease_category'],
    y=moderate_data['patients'],

    name='<b>Moderate</b>',
    marker=dict(color='dodgerblue'),

    text=moderate_data['patients'],
    texttemplate='<b>%{text}</b>',
    textposition='inside',

    hovertemplate=
    '<b>Severity:</b> Moderate<br>' +
    '<b>Patients:</b> %{y}<br><extra></extra>'
    ))
severe_data = severity_analysis[severity_analysis['severity_level'] == 'Severe']

fig.add_trace(go.Bar(
    x=severe_data['disease_category'],
    y=severe_data['patients'],

    name='<b>Severe</b>',

    text=severe_data['patients'],
    texttemplate='<b>%{text}</b>',
    textposition='outside',

    marker=dict(color='blueviolet'),

    hovertemplate=
    '<b>Severity:</b> Severe<br>' +
    '<b>Patients:</b> %{y}<br><extra></extra>'
    ))
fig.update_layout(

    title=dict(
        text='<b>Patient Volume by Disease Category & Severity</b>',
        x=0.5,
        font=dict(size=24)),
        
    xaxis=dict(
        title=dict(
            text='<b>Disease Category</b>',
            font=dict(size=18)),
        tickfont=dict(size=14)),

    yaxis=dict(
        title=dict(
            text='<b>Number of Patients</b>',
            font=dict(size=18)),
        tickfont=dict(size=14),
        showgrid=True,
        gridcolor='rgba(0,0,0,0.11)'),

    barmode='stack',
    bargap=0.4,
    template='simple_white',
    hovermode='x unified',
    margin=dict(l=90, r=0, t=100, b=80),

    legend=dict(
        title=dict(text='<b>Severity Level</b>', font=dict(size=14)),
        x=0.846,
        y=1.14,
        bgcolor='rgba(255,255,255,0.7)',
        bordercolor='black',
        borderwidth=1
        ))
fig.show()

### 3) Disease & Severity Analysis

    Most common diseases and their severity distribution

In [74]:
disease_severity = df.groupby(['disease', 'severity_level']).size().reset_index(name='patients')

top_diseases = df['disease'].value_counts().head(10).index

disease_severity = disease_severity[disease_severity['disease'].isin(top_diseases)]

disease_severity.head()

,disease,severity_level,patients
0,Allergic Rhinitis,Mild,60
1,Allergic Rhinitis,Moderate,39
2,Allergic Rhinitis,Severe,11
3,Arthritis,Mild,58
4,Arthritis,Moderate,30


In [75]:
fig = go.Figure()

mild = disease_severity[disease_severity['severity_level'] == 'Mild']

fig.add_trace(go.Bar(
    x=mild['disease'],
    y=mild['patients'],

    name='<b>Mild</b>',
    marker=dict(color='cyan'),

    text=mild['patients'],
    texttemplate='<b>%{text}</b>',
    textposition='inside',

    hovertemplate=
    '<b>Severity:</b> Mild<br>' +
    '<b>Patients:</b> %{y}<extra></extra>'
    ))
moderate = disease_severity[disease_severity['severity_level'] == 'Moderate']

fig.add_trace(go.Bar(
    x=moderate['disease'],
    y=moderate['patients'],

    name='<b>Moderate</b>',
    marker=dict(color='dodgerblue'),

    text=moderate['patients'],
    texttemplate='<b>%{text}</b>',
    textposition='inside',

    hovertemplate=
    '<b>Severity:</b> Moderate<br>' +
    '<b>Patients:</b> %{y}<br><extra></extra>'
    ))
severe = disease_severity[disease_severity['severity_level'] == 'Severe']

fig.add_trace(go.Bar(
    x=severe['disease'],
    y=severe['patients'],

    name='<b>Severe</b>',
    marker=dict(color='blueviolet'),

    text=severe['patients'],
    texttemplate='<b>%{text}</b>',
    textposition='outside',

    hovertemplate=
    '<b>Severity:</b> Severe<br>' +
    '<b>Patients:</b> %{y}<br><extra></extra>'
    ))
fig.update_layout(

    title=dict(
        text='<b>Most Common Diseases & Severity Distribution</b>',
        x=0.5,
        font=dict(size=24)),

    xaxis=dict(
        title=dict(
            text='<b>Disease</b>',
            font=dict(size=18)),
        tickfont=dict(size=14),
        tickangle=-15),

    yaxis=dict(
        title=dict(
            text='<b>Number of Patients</b>',
            font=dict(size=16)),
        showgrid=True,
        tickfont=dict(size=14),
        gridcolor='rgba(0,0,0,0.18)'),

    barmode='stack',
    template='simple_white',
    hovermode='x unified',
    margin=dict(l=80, r=20, t=100, b=80),
    bargap=0.25,

    legend=dict(
        title=dict(text='<b>Severity Level</b>',font=dict(size=14)),
        x=0.88,
        y=1.25,
        bgcolor='rgba(255,255,255,0.7)',
        bordercolor='black',
        borderwidth=1
    ))
fig.show()

    Default rate / repeat visit rate by Disease Category and Severity Level

In [76]:
df['repeat_visit'] = (df['number_of_visits'] > 1).astype(int)

visit_analysis = df.groupby(['disease_category', 'severity_level']).agg(

    total_patients=('patientid', 'count'),

    repeat_visit_rate=('repeat_visit',lambda x: x.mean() * 100),

    avg_visits=('number_of_visits','mean')
    
).reset_index()

visit_analysis

,disease_category,severity_level,total_patients,repeat_visit_rate,avg_visits
0,Acute,Mild,84,66.666667,2.321429
1,Acute,Moderate,71,69.014085,2.366197
2,Acute,Severe,19,73.684211,2.684211
3,Cardiac,Mild,56,94.642857,5.928571
4,Cardiac,Moderate,36,97.222222,6.222222
5,Cardiac,Severe,13,84.615385,5.000000
6,Chronic,Mild,212,100.000000,6.985849
7,Chronic,Moderate,137,100.000000,6.978102
8,Chronic,Severe,47,100.000000,7.617021
9,Neurology,Mild,64,73.437500,3.078125


In [99]:
fig = go.Figure()

mild = visit_analysis[visit_analysis['severity_level'] == 'Mild']

fig.add_trace(go.Bar(
    x=mild['disease_category'],
    y=mild['repeat_visit_rate'],

    name='<b>Mild</b>',
    text=mild['repeat_visit_rate'].round(1),
    texttemplate='<b>%{text}%</b>',
    textposition='inside',
    marker=dict(color='#0A2540'),

    customdata=mild[['total_patients', 'avg_visits']],

    hovertemplate=
    '<b>Severity:</b> Mild<br>' +
    '<b>Repeat Visit Rate:</b> %{y:.2f}%<br>' +
    '<b>Total Patients:</b> %{customdata[0]}<br>' +
    '<b>Avg Visits:</b> %{customdata[1]:.1f}<br><extra></extra>'
))
moderate = visit_analysis[visit_analysis['severity_level'] == 'Moderate']

fig.add_trace(go.Bar(
    x=moderate['disease_category'],
    y=moderate['repeat_visit_rate'],

    name='<b>Moderate</b>',
    marker=dict(color='chartreuse'),

    text=moderate['repeat_visit_rate'].round(1),
    texttemplate='<b>%{text}%</b>',
    textposition='inside',
    textfont=dict(color='black'),

    customdata=moderate[['total_patients', 'avg_visits']],
    hovertemplate=
    '<b>Severity:</b> Moderate<br>' +
    '<b>Repeat Visit Rate:</b> %{y:.2f}%<br>' +
    '<b>Total Patients:</b> %{customdata[0]}<br>' +
    '<b>Avg Visits:</b> %{customdata[1]:.1f}<br><extra></extra>'
))
severe = visit_analysis[visit_analysis['severity_level'] == 'Severe']

fig.add_trace(go.Bar(
    x=severe['disease_category'],
    y=severe['repeat_visit_rate'],

    name='<b>Severe</b>',
    marker=dict(color='lightcoral'),

    text=severe['repeat_visit_rate'].round(1),
    texttemplate='<b>%{text}%</b>',
    textposition='inside',
    textfont=dict(color='white'),

    customdata=severe[['total_patients', 'avg_visits']],
    hovertemplate=
    '<b>Severity:</b> Severe<br>' +
    '<b>Repeat Visit Rate:</b> %{y:.2f}%<br>' +
    '<b>Total Patients:</b> %{customdata[0]}<br>' +
    '<b>Avg Visits:</b> %{customdata[1]:.1f}<extra></extra>'
))
fig.update_layout(

    title=dict(
        text='<b>Repeat Visit Rate by Disease Category & Severity</b>',
        x=0.5,
        font=dict(size=20)),

    xaxis=dict(
        title=dict(
            text='<b>Disease Category</b>', 
            font=dict(size=18)),
        tickfont=dict(size=14)),

    yaxis=dict(
        title=dict(
            text='<b>Repeat Visit Rate (%)</b>', 
            font=dict(size=18)),
        tickfont=dict(size=14),
        ticksuffix='%',
        showgrid=True,
        gridcolor='rgba(0,0,0,0.14)'),

    barmode='group',
    template='simple_white',
    hovermode='x unified',
    margin=dict(l=100, r=20, t=100, b=80),

    legend=dict(
        title=dict(text='<b>Severity Level</b>', font=dict(size=14)),
        orientation='h',
        x=0.32,
        y=1.1,
        bgcolor='rgba(255,255,255,0.7)',
        bordercolor='lightgray',
        borderwidth=1
        ))

fig.show()

    Average medicine cost by Disease Category and Severity

In [78]:

medicine_cost = df.groupby(['disease_category', 'severity_level']).agg(
        
        avg_cost=('cost_price', 'mean'),

        total_patients=('patientid', 'count'),

        avg_quantity=('quantity', 'mean')

).reset_index()

medicine_cost['avg_cost'] = medicine_cost['avg_cost'].round(0)

medicine_cost.head()

,disease_category,severity_level,avg_cost,total_patients,avg_quantity
0,Acute,Mild,2921.0,84,3.119048
1,Acute,Moderate,3071.0,71,2.901408
2,Acute,Severe,3666.0,19,3.052632
3,Cardiac,Mild,2723.0,56,3.017857
4,Cardiac,Moderate,2955.0,36,3.055556


In [79]:
fig = go.Figure()

mild = medicine_cost[medicine_cost['severity_level'] == 'Mild']

fig.add_trace(go.Bar(
    x=mild['disease_category'],
    y=mild['avg_cost'],

    name='<b>Mild</b>',
    marker=dict(color='#0A2540'),

    text=mild['avg_cost'].apply(lambda x: f"${x:,.0f}"),
    texttemplate='<b>%{text}</b>',
    textposition='outside',

    customdata=mild[['total_patients', 'avg_quantity']],
    hovertemplate=
    '<b>Severity:</b> Mild<br>' +
    '<b>Average Cost:</b> $%{y:,.0f}<br>' +
    '<b>Total Patients:</b> %{customdata[0]}<br>' +
    '<b>Avg Quantity:</b> %{customdata[1]:.1f}<br><extra></extra>'
    ))
moderate = medicine_cost[medicine_cost['severity_level'] == 'Moderate']

fig.add_trace(go.Bar(
    x=moderate['disease_category'],
    y=moderate['avg_cost'],

    name='<b>Moderate</b>',
    marker=dict(color='chartreuse'),

    text=moderate['avg_cost'].apply(lambda x: f"${x:,.0f}"),
    texttemplate='<b>%{text}</b>',
    textposition='outside',

    customdata=moderate[['total_patients', 'avg_quantity']],
    hovertemplate=
    '<b>Severity:</b> Moderate<br>' +
    '<b>Average Cost:</b> $%{y:,.0f}<br>' +
    '<b>Total Patients:</b> %{customdata[0]}<br>' +
    '<b>Avg Quantity:</b> %{customdata[1]:.1f}<br><extra></extra>'
    ))
severe = medicine_cost[medicine_cost['severity_level'] == 'Severe']

fig.add_trace(go.Bar(
    x=severe['disease_category'],
    y=severe['avg_cost'],

    name='<b>Severe</b>',
    marker=dict(color='deepskyblue'),

    text=severe['avg_cost'].apply(lambda x: f"${x:,.0f}"),
    texttemplate='<b>%{text}</b>',
    textposition='outside',

    customdata=severe[['total_patients', 'avg_quantity']],
    hovertemplate=
    '<b>Severity:</b> Severe<br>' +
    '<b>Average Cost:</b> $%{y:,.0f}<br>' +
    '<b>Total Patients:</b> %{customdata[0]}<br>' +
    '<b>Avg Quantity:</b> %{customdata[1]:.1f}<extra></extra>'
    ))
fig.update_layout(
    title=dict(
        text='<b>Average Medicine Cost by Disease Category & Severity</b>',
        x=0.5,
        font=dict(size=24)),

    xaxis=dict(
        title=dict(
            text='<b>Disease Category</b>',
            font=dict(size=18)),
            tickfont=dict(size=16)),

    yaxis=dict(
        title=dict(
            text='<b>Average Medicine Cost ($)</b>',
            font=dict(size=16)),
        tickfont=dict(size=14),
        tickformat=',.0f',
        tickprefix='$',
        showgrid=True,
        gridcolor='rgba(0,0,0,0.18)'),

    barmode='group',
    bargap=0.22,
    template='simple_white',
    hovermode='x unified',
    margin=dict(l=110, r=20, t=100, b=80),

    legend=dict(
        title=dict(text='<b>Severity Level</b>', font=dict(size=14)),
        x=0.85,
        y=1.2,
        bgcolor='rgba(255,255,255,0.7)',
        bordercolor='black',
        borderwidth=1
        ))
fig.show()

### 4) Treatment & Cost Insights

    Top 10 most prescribed medicines by quantity and total cost

In [80]:
medicine_analysis = df.groupby('medicinename').agg(

    total_quantity=('quantity', 'sum'),

    total_cost=('total_medicine_cost', 'sum')

).reset_index()

medicine_analysis = medicine_analysis.sort_values(by=['total_quantity', 'total_cost'], ascending=False).head(10)

medicine_analysis['total_cost'] = medicine_analysis['total_cost'].apply(lambda x: f"${x:,.0f}")

medicine_analysis

,medicinename,total_quantity,total_cost
5,Medicine_13,44,"$34,584"
3,Medicine_11,40,"$175,840"
36,Medicine_41,39,"$118,326"
66,Medicine_69,39,"$61,269"
80,Medicine_81,39,"$43,368"
46,Medicine_50,38,"$185,782"
16,Medicine_23,38,"$163,400"
32,Medicine_38,38,"$153,444"
6,Medicine_14,38,"$96,634"
47,Medicine_51,37,"$177,674"


In [81]:
medicine_analysis = df.groupby('medicinename').agg(
    total_quantity=('quantity', 'sum'),
    total_cost=('total_medicine_cost', 'sum')
).reset_index()

medicine_analysis = medicine_analysis.sort_values(by=['total_quantity', 'total_cost'], ascending=False).head(10)

colors = ['green' if x > 119000 else 'red' for x in medicine_analysis['total_cost']]

fig = make_subplots(specs=[[{"secondary_y": True}]])

fig.add_trace(go.Bar(
    x=medicine_analysis['medicinename'],
    y=medicine_analysis['total_quantity'],

    name='<b>Total Quantity</b>',
    text=medicine_analysis['total_quantity'],
    texttemplate='<b>%{text}</b>',
    textposition='outside',
    textfont=dict(size=14, color='black'),

    marker=dict(
        color=medicine_analysis['total_quantity'],
        colorscale='twilight_r',
        line=dict(color='white', width=1.5)),

    hovertemplate=
    '<b>Medicine:</b> %{x}<br>' +
    '<b>Total Quantity:</b> %{y:,}<br><extra></extra>'

), secondary_y=False)

fig.add_trace(go.Scatter(
    x=medicine_analysis['medicinename'],
    y=medicine_analysis['total_cost'],

    mode='lines+markers+text',
    name='<b>Total Cost</b>',

    line=dict(
        color='#1f77b4',
        width=4,
        shape='spline'),

    marker=dict(
        size=9,
        color=colors,
        line=dict(color='white', width=1.5)),

    hovertemplate=
    '<b>Medicine:</b> %{x}<br>' +
    '<b>Total Cost:</b> $%{y:,}<extra></extra>'

), secondary_y=True)

fig.update_layout(
    title=dict(
        text='<b>Top 10 Most Prescribed Medicines</b>',
        x=0.5,
        font=dict(size=24)),

    template='simple_white',
    hovermode='x unified',
    bargap=0.25,
    margin=dict(t=90, l=80, r=20, b=100),

    legend=dict(
        x=0.78,
        y=1.15,
        bgcolor='rgba(255,255,255,0.7)',
        bordercolor='lightgray',
        borderwidth=1))

fig.update_xaxes(title=dict(text='<b>Medicines Name</b>', font=dict(size=16)), tickangle=15, tickfont=dict(size=14))
fig.update_yaxes(title=dict(text='<b>Total Quantity</b>', font=dict(size=16)), tickfont=dict(size=14), secondary_y=False)
fig.update_yaxes(title=dict(text='<b>Total Cost ($K)</b>', font=dict(size=16)), tickfont=dict(size=14), secondary_y=True)

fig.show()

    Average medicine cost per patient by Age Group and Gender

In [82]:
per_patient_cost = df.groupby(['age_group','gender']).agg(

    avg_cost=('total_medicine_cost', 'mean'),

    total_patients=('patientid', 'nunique')
    
).reset_index()

per_patient_cost['avg_cost_k'] = per_patient_cost['avg_cost'] / 1000

per_patient_cost

,age_group,gender,avg_cost,total_patients,avg_cost_k
0,18-30,Female,8969.950000,1,8.969950
1,18-30,Male,9027.511111,9,9.027511
2,31-45,Female,9266.764286,7,9.266764
3,31-45,Male,9285.650000,5,9.285650
4,46-60,Female,9390.930000,5,9.390930
5,46-60,Male,9389.431250,8,9.389431
6,61-80,Female,9258.437500,4,9.258437
7,61-80,Male,9200.575000,10,9.200575


In [83]:
fig = go.Figure()

male_data = per_patient_cost[per_patient_cost['gender'] == 'Male']

fig.add_trace(go.Bar(
    x=male_data['age_group'],
    y=male_data['avg_cost_k'],

    name='<b>Male</b>',

    textfont=dict(size=15),
    text=male_data['avg_cost_k'].round(1),
    texttemplate='<b>%{text}K</b>',
    textposition='outside',

    marker=dict(
        color='#1E88E5',
        line=dict(
            color='white', 
            width=1.5)),

    customdata=male_data['total_patients'],
    hovertemplate=
    '<b>Gender:</b> Male<br>' +
    '<b>Age Group:</b> %{x}<br>' +
    '<b>Avg Cost:</b> $%{y:.2f}K<br>' +
    '<b>Total Patients:</b> %{customdata:,}<extra></extra>'
    ))
female_data = per_patient_cost[per_patient_cost['gender'] == 'Female']

fig.add_trace(go.Bar(
    x=female_data['age_group'],
    y=female_data['avg_cost_k'],

    name='<b>Female</b>',

    textfont=dict(size=15),
    text=female_data['avg_cost_k'].round(1),
    texttemplate='<b>%{text}K</b>',
    textposition='outside',

    marker=dict(
        color='#E91E63',
        line=dict(
            color='white', 
            width=1.5)),

    customdata=female_data['total_patients'],
    hovertemplate=
    '<b>Gender:</b> Female<br>' +
    '<b>Age Group:</b> %{x}<br>' +
    '<b>Avg Cost:</b> $%{y:.2f}K<br>' +
    '<b>Total Patients:</b> %{customdata:,}<extra></extra>'
    ))
fig.update_layout(
    title=dict(
        text='<b>Average Medicine Cost per Patient by Age Group & Gender</b>',
        x=0.5,
        font=dict(size=24)),

    xaxis=dict(
        title=dict(
            text='<b>Age Group & Gender</b>',
            font=dict(size=18)),
        tickfont=dict(size=14)),

    yaxis=dict(
        title=dict(
            text='<b>Average Medicine Cost ($K)</b>',
            font=dict(size=18)),
        tickfont=dict(size=14),
        ticksuffix='K',
        showgrid=True,
        gridcolor='rgba(0,0,0,0.18)'),

    barmode='group',
    template='simple_white',
    bargap=0.22,
    hovermode='x unified',

    legend=dict(
        x=0.9,
        y=1.2,
        bgcolor='rgba(255,255,255,0.7)',
        bordercolor='lightgray',
        borderwidth=1
    ))
fig.show()

    Relationship between Severity Level and Total_Medicine_Cost (box plot)

In [84]:
fig = go.Figure()

mild = df[df['severity_level'] == 'Mild']

fig.add_trace(go.Box(
    y=mild['total_medicine_cost'],

    name='Mild',
    boxmean=True,
    marker=dict(color='#43A047'),

    hovertemplate=
    '<b>Severity:</b> Mild<br>' +
    '<b>Medicine Cost:</b> $%{y:,.0f}<extra></extra>'
    ))

moderate = df[df['severity_level'] == 'Moderate']

fig.add_trace(go.Box(
    y=moderate['total_medicine_cost'],

    name='Moderate',
    boxmean=True,
    marker=dict(color='#FB8C00'),

    hovertemplate=
    '<b>Severity:</b> Moderate<br>' +
    '<b>Medicine Cost:</b> $%{y:,.0f}<extra></extra>'
    ))

severe = df[df['severity_level'] == 'Severe']

fig.add_trace(go.Box(
    y=severe['total_medicine_cost'],

    name='Severe',
    boxmean=True,
    marker=dict(color='#E53935'),

    hovertemplate=
    '<b>Severity:</b> Severe<br>' +
    '<b>Medicine Cost:</b> $%{y:,.0f}<extra></extra>'
    ))
fig.update_layout(
    title=dict(
        text='<b>Severity Level vs Total Medicine Cost</b>',
        x=0.5,
        font=dict(size=24)),

    xaxis=dict(
        title=dict(
            text='<b>Severity Level</b>',
            font=dict(size=18)),
        tickfont=dict(size=16)),

    yaxis=dict(
        title=dict(
            text='<b>Total Medicine Cost ($)</b>',
            font=dict(size=16)),
        tickfont=dict(size=14),
        showgrid=True,
        gridcolor='rgba(0,0,0,0.18)'),

    template='simple_white',
    boxmode='group',
    hovermode='closest',

    legend=dict(
        orientation='h',
        x=0.38,
        y=1.1
    ))
fig.show()

### 5) Doctor & Hospital Performance

    Number of patients and average visits by Doctor Specialty

In [85]:
doctor_analysis = df.groupby('doctor_specialty').agg(

    no_of_patients=('patientid', 'nunique'),

    avg_visits=('number_of_visits', 'mean')

    ).reset_index()

doctor_analysis['avg_visits'] = doctor_analysis['avg_visits'].round(1)
doctor_analysis = doctor_analysis.sort_values(by='no_of_patients',ascending=False)

doctor_analysis

,doctor_specialty,no_of_patients,avg_visits
0,Cardiology,50,6.5
2,Endocrinology,49,7.1
3,General Medicine,49,2.4
6,Rheumatology,47,6.9
4,Neurology,45,3.1
5,Pulmonology,43,4.7
1,ENT,42,4.5


In [86]:
fig = make_subplots(specs=[[{"secondary_y": True}]])

fig.add_trace(go.Bar(

    x=doctor_analysis['doctor_specialty'],
    y=doctor_analysis['no_of_patients'],

    name='<b>Patients</b>',

    text=doctor_analysis['no_of_patients'],
    texttemplate='<b>%{text:,}</b>',
    textposition='outside',
    textfont=dict(size=14),

    marker=dict(
        color=doctor_analysis['no_of_patients'],
        colorscale='temps',
        line=dict(color='white', width=1.5)
        ),
        
    hovertemplate=
    '<b>Specialty:</b> %{x}<br>' +
    '<b>Patients:</b> %{y:,}<br><extra></extra>' 

    ), secondary_y=False)

fig.add_trace(go.Scatter(

    x=doctor_analysis['doctor_specialty'],
    y=doctor_analysis['avg_visits'],
    
    name='<b>Average Visits</b>',
    mode='lines+markers+text',

    line=dict(
        color='deepskyblue',
        width=4,
        shape='spline'
    ),
    marker=dict(
        size=9,
        color='#0A2540',
        line=dict(color='white', width=2)
    ),
    fill = 'tozeroy',
    fillcolor='rgba(30,136,229,0.15)',
    hovertemplate=
    '<b>Specialty:</b> %{x}<br>' +
    '<b>Average Visits:</b> %{y:.1f}<extra></extra>'

    ), secondary_y=True)

fig.update_layout(
    title=dict(
        text='<b>Doctor Specialty: Patients vs Average Visits</b>',
        x=0.5,
        font=dict(size=24)
        ),
    template='simple_white',
    hovermode='x unified',
    legend=dict(
        x=0.8,
        y=1.2,
        bgcolor='rgba(255,255,255,0.7)'
        ),
    margin=dict(t=90, l=80, r=0, b=80)
    )

fig.update_xaxes(
    title=dict(text='<b>Doctor Specialty</b>', font=dict(size=18)),
    tickfont=dict(size=14),
    )
fig.update_yaxes(
    title=dict(text='<b>Number of Patients</b>', font=dict(size=18)),
    tickfont=dict(size=14),
    secondary_y=False
    )
fig.update_yaxes(
    title=dict(text='<b>Average Visits</b>', font=dict(size=18)),
    tickfont=dict(size=14),
    secondary_y=True
    )
fig.show()

    Top performing vs high-cost specialties (volume vs average cost)

In [87]:
specialty_analysis = df.groupby('doctor_specialty').agg(
    
    patient_count=('patientid','nunique'),

    avg_cost=('total_medicine_cost','mean'),

    total_cost=('total_medicine_cost','sum'),

    avg_visits=('number_of_visits','mean')

    ).reset_index()

specialty_analysis['avg_cost'] = specialty_analysis['avg_cost'].round(0)
specialty_analysis['avg_visits'] = specialty_analysis['avg_visits'].round(1)

specialty_analysis

,doctor_specialty,patient_count,avg_cost,total_cost,avg_visits
0,Cardiology,50,9101.0,1874789,6.5
1,ENT,42,8591.0,945038,4.5
2,Endocrinology,49,9600.0,1891119,7.1
3,General Medicine,49,9333.0,1623926,2.4
4,Neurology,45,9257.0,962695,3.1
5,Pulmonology,43,8982.0,997040,4.7
6,Rheumatology,47,9408.0,922019,6.9


In [88]:
fig = go.Figure()

fig.add_trace(go.Scatter(

    x=specialty_analysis['patient_count'],
    y=specialty_analysis['avg_cost'],

    mode='markers+text',
    text=specialty_analysis['doctor_specialty'],
    textposition='top center',

    marker=dict(
        size=specialty_analysis['total_cost'] / 40000,
        opacity=0.85,

        color=specialty_analysis['total_cost'],
        colorscale='temps',
        showscale=True,
        colorbar=dict(title=dict(text='<b>Total Cost</b>', font=dict(size=16))),

        line=dict(color='white',width=2)),

    customdata=specialty_analysis[['total_cost','avg_visits']],
    hovertemplate=
    '<b>Specialty:</b> %{text}<br>' +
    '<b>Patients:</b> %{x:,}<br>' +
    '<b>Total Cost:</b> $%{customdata[0]:,.0f}<br>' +
    '<b>Average Cost:</b> $%{y:,.0f}<br>' +
    '<b>Average Visits:</b> %{customdata[1]:.1f}<extra></extra>'
    ))
fig.update_layout(

    title=dict(
        text='<b>Top Performing vs High-Cost Specialties</b>', 
        x=0.5, 
        font=dict(size=24)
        ),
    xaxis=dict(
        title=dict(text='<b>Patient Volume</b>', font=dict(size=16)),
        tickfont=dict(size=14),
        showgrid=True,
        gridcolor='rgba(0,0,0,0.08)'
        ),
    yaxis=dict(
        title=dict(text='<b>Average Medicine Cost ($)</b>', font=dict(size=16)),
        tickfont=dict(size=14),
        showgrid=True,
        gridcolor='rgba(0,0,0,0.08)'
        ),
    margin=dict(t=90, l=95, r=70, b=70),
    template='simple_white',
    hovermode='closest'
    )
fig.show()

    Hospital/Clinic wise patient load and repeat visit rate

In [89]:
hospital_analysis = df.groupby('hospital/clinic').agg(

    patient_load=('patientid','nunique'),

    repeat_visit_rate=('number_of_visits', lambda x: (x > 1).mean() * 100),

    avg_visits=('number_of_visits','mean')

    ).reset_index()

hospital_analysis['repeat_visit_rate'] = hospital_analysis['repeat_visit_rate'].round(1)
hospital_analysis['avg_visits'] = hospital_analysis['avg_visits'].round(1)
hospital_analysis = hospital_analysis.sort_values(by='patient_load',ascending=False)

hospital_analysis

,hospital/clinic,patient_load,repeat_visit_rate,avg_visits
3,Greenfield Medical Centre,49,88.8,5.4
0,Apex Hospitals,47,89.3,5.0
6,Riverfront Clinic,47,86.5,5.4
8,Unity Care,47,87.2,4.8
4,Lotus Hospital,46,92.5,5.1
5,Metro Health Clinic,46,90.5,5.0
1,City Care Hospital,45,92.8,5.6
2,Fortune Care,44,85.7,4.9
7,Sunrise Multispeciality,44,87.5,5.4


In [90]:
fig = make_subplots(specs=[[{"secondary_y": True}]])

text_colors = [
    'whitesmoke' if x > hospital_analysis['patient_load'].median()
    else 'black'
    for x in hospital_analysis['patient_load']]

fig.add_trace(go.Bar(

    x=hospital_analysis['hospital/clinic'],
    y=hospital_analysis['patient_load'],

    name='<b>Patient Load</b>',
    text=hospital_analysis['patient_load'],

    texttemplate='<b>%{text:,}</b>',
    textposition='inside',
    textfont=dict(size=14, color=text_colors),

    marker=dict(
        color=hospital_analysis['patient_load'],
        colorscale='ylorrd',
        line=dict(color='white', width=1.5))
    ), secondary_y=False)

fig.add_trace(go.Scatter(

    x=hospital_analysis['hospital/clinic'],
    y=hospital_analysis['repeat_visit_rate'],

    name='<b>Repeat Visit Rate</b>',
    mode='lines+markers+text',

    text=hospital_analysis['repeat_visit_rate'].round(1),
    texttemplate='<b>%{text}%</b>',
    textposition='top center',
    textfont=dict(size=14,color=text_colors),

    line=dict(
        color='cyan',
        width=4,
        shape='spline'),

    marker=dict(
        size=9,
        color='#0A2540',
        line=dict(color='white', width=2)),
        

    hovertemplate=
    '<b>Hospital:</b> %{x}<br>' +
    '<b>Repeat Visit Rate:</b> %{y:.1f}%<extra></extra>'
    ), secondary_y=True)

fig.update_layout(

    title=dict( text='<b>Patient Load vs Repeat Visit Rate</b>', x=0.5, font=dict(size=24)),

    template='simple_white',
    hovermode='x unified',
    bargap=0.25,
    margin=dict(r=0),

    legend=dict(
        x=0.88,
        y=1.22,
        bgcolor='rgba(255,255,255,0.7)'
    ))
fig.update_xaxes(
    title=dict(text='<b>Hospital / Clinic</b>',font=dict(size=18)),tickangle=15,tickfont=dict(size=14))

fig.update_yaxes(
    title=dict(text='<b>Patient Load</b>',font=dict(size=18)),tickangle=15,tickfont=dict(size=14),secondary_y=False)

fig.update_yaxes(
    title=dict(text='<b>Repeat Visit Rate (%)</b>',font=dict(size=18)),tickangle=15,tickfont=dict(size=14),ticksuffix='%',secondary_y=True)
    
fig.show()

### 6) Visit Patterns & Efficiency

    Distribution of Number of Visits (FY 2024)

In [91]:
visit_dist = df['number_of_visits'].value_counts().reset_index()

visit_dist.columns = ['number_of_visits', 'patients']

visit_dist = visit_dist.sort_values(by='number_of_visits')

visit_dist

,number_of_visits,patients
3,1,110
1,2,135
2,3,111
0,4,158
4,5,83
7,6,71
6,7,73
5,8,80
8,9,55
9,10,54


In [92]:
fig = make_subplots(
    rows=2, cols=1,

    shared_xaxes=True,
    row_heights=[0.18,0.82],
    vertical_spacing=0.04
)
fig.add_trace(go.Box(
    x=df['number_of_visits'],
    
    name='',
    marker=dict(color='#1E88E5'),
    boxmean=True,

    hovertemplate=
    '<b>Visits:</b> %{x}<extra></extra>'

), row=1, col=1)

fig.add_trace(go.Histogram(

    x=df['number_of_visits'],

    nbinsx=12,
    texttemplate='<b>%{y}</b>',
    textposition='outside',

    marker=dict(
        color='#1E88E5',
        line=dict(
            color='white',
            width=1.5
        )),
    hovertemplate=
    '<b>Visits:</b> %{x}<br>' +
    '<b>Patients:</b> %{y:,}<extra></extra>'

), row=2, col=1)

avg_patients = visit_dist['patients'].mean()

fig.add_hline(

    y=avg_patients,

    line_dash='dash',
    line_color='black',  
    line_width=2,           
    opacity=1,              
    layer='above',

    annotation_text=f'<b>Avg Patients: {avg_patients:.1f}</b>',
    annotation_position='top right',
    annotation_font=dict(color='black',size=14),

    row=2, col=1
    )
fig.update_layout(

    title=dict(
        text='<b>Distribution of Patient Visits</b>', x=0.5, font=dict(size=24)),

    template='simple_white',
    bargap=0.12,
    showlegend=False
    )
fig.update_xaxes(
    title=dict(text='<b>Number of Visits</b>',font=dict(size=16)),tickfont=dict(size=14),
    dtick=1,
    row=2,
    col=1
)
fig.update_yaxes(
    title=dict(text='<b>Patient Count</b>',font=dict(size=16)),tickfont=dict(size=14),
    range=[0, visit_dist['patients'].max() * 1.2],
    row=2,
    col=1
)
fig.show()

    Correlation between Age, Severity, and Visit Frequency

In [93]:
# Severity Encoding
severity_map = {'Mild':1,'Moderate':2,'Severe':3}

df['severity_num'] = df['severity_level'].map(severity_map)

corr_data = df[['age','severity_num','number_of_visits']].corr().round(2)

corr_data

,age,severity_num,number_of_visits
age,1.00,0.04,0.05
severity_num,0.04,1.00,0.02
number_of_visits,0.05,0.02,1.00


In [94]:
fig = go.Figure(go.Heatmap(

    z=corr_data.values,
    x=corr_data.columns,
    y=corr_data.columns,

    colorscale='RdBu',
    zmin=-1,
    zmax=1,
    text=corr_data.round(2),
    texttemplate='<b>%{text}</b>',

    hovertemplate =
        '<b>%{x} vs %{y}</b><br>' +
        'Correlation: %{z:.2f}<extra></extra>'
))

fig.update_layout(

    title=dict(
        text='<b>Correlation: Age, Severity & Visit Frequency</b>', font=dict(size=20), x=0.5),

        template='simple_white')

fig.update_xaxes(tickfont=dict(size=14))
fig.update_yaxes(tickfont=dict(size=14))

fig.show()

    Patients with high repeat visits (>8 visits) — what diseases/age groups dominate?

In [95]:
high_repeat = df[df['number_of_visits'] > 8]

repeat_analysis = high_repeat.groupby(['disease','age_group']).agg(patient_count=('patientid','nunique')).reset_index()

repeat_analysis

,disease,age_group,patient_count
0,Arthritis,18-30,8
1,Arthritis,31-45,1
2,Arthritis,46-60,7
3,Arthritis,61-80,9
4,Coronary Artery Disease,18-30,2
5,Coronary Artery Disease,31-45,5
6,Coronary Artery Disease,46-60,4
7,Coronary Artery Disease,61-80,8
8,Diabetes,18-30,5
9,Diabetes,31-45,5


In [96]:
fig = go.Figure()

age_groups = repeat_analysis['age_group'].unique()

colors = [
    '#1E88E5',
    '#43A047',
    '#FB8C00',
    '#E53935'
]

for age,color in zip(age_groups,colors):

    temp = repeat_analysis[repeat_analysis['age_group'] == age]

    fig.add_trace(go.Bar(

        x=temp['disease'],
        y=temp['patient_count'],

        name=f'<b>{age}</b>',
        marker=dict(color=color),

        text=temp['patient_count'],
        texttemplate='<b>%{text}</b>',
        textposition='outside',

        hovertemplate=
        '<b>Disease:</b> %{x}<br>' +
        '<b>Age Group:</b> '+str(age)+'<br>' +
        '<b>Patients:</b> %{y:,}<extra></extra>'
    ))

fig.update_layout(

    title=dict(
        text='<b>Highest Repeated Patients (Age & Disease)</b>',x=0.5,font=dict(size=24)),

    xaxis=dict(
        title=dict(text='<b>Disease</b>',font=dict(size=16)),tickfont=dict(size=14)
    ),

    yaxis=dict(
        title=dict(text='<b>Number of Patients</b>',font=dict(size=16)),tickfont=dict(size=14),
        showgrid=True,gridcolor='rgba(0,0,0,0.15)',range=[0, repeat_analysis['patient_count'].max() + 1]
    ),

    barmode='group',
    template='simple_white',
    bargap=0.22,

    legend=dict(
        title=dict(text='<b>Age Group</b>',font=dict(size=16)),
        orientation='v',
        x=0.89,
        y=1.18
    )
)

fig.show()

## Business Recommendations(Suggestions)

### 1. Strengthen Chronic Disease Management Programs

Chronic diseases account for the highest patient volume and treatment expenditure. Healthcare providers should invest in:

- Preventive care programs
- Regular monitoring systems
- Patient education initiatives
- Digital follow-up services

This can reduce long-term treatment costs and improve patient outcomes.

### 2. Focus on High-Cost Patient Segments

The analysis shows that Chronic-Mild and Chronic-Moderate patient groups contribute the largest medicine expenditures.

    Recommendations:

- Implement medication optimization programs.
- Review prescription patterns regularly.
- Negotiate better pricing with pharmaceutical suppliers.
- Introduce personalized treatment plans for high-cost patients.

### 3. Improve Resource Allocation for High-Demand Specialties

    Specialties such as:

- Endocrinology
- Rheumatology
- Cardiology

show higher patient volumes and visit frequencies.

    Healthcare organizations should:

- Allocate additional staff and resources.
- Optimize appointment scheduling.
- Expand specialist availability during peak demand periods.

### 4. Strengthen Patient Retention and Follow-Up Strategies

Repeat visit rates exceed 90% in several disease categories, particularly chronic conditions.

    Suggested actions:

- Develop patient engagement programs.
- Use automated appointment reminders.
- Introduce telemedicine consultations.
- Create long-term care plans for recurring patients.

### 5. Develop Targeted Care Programs for Frequent Visitors

Patients with more than 8 annual visits are concentrated in diseases such as:

- Arthritis
- Diabetes
- Hypertension
- Coronary Artery Disease

Specialized disease management programs should be developed to improve treatment effectiveness and reduce unnecessary repeat visits.